# 013a01: Online Model Latency & Reliability Test

Minimal notebook to measure run-to-run variability of `:online` model queries.
Runs each model multiple times on the same question to see if timeouts are
a pattern or just bad luck.

**Parent notebook**: 013a (Multi-Model Base Rate Researcher)

In [1]:
# Inputs

# --- Models to test (OpenRouter paths, :online appended automatically) ---
MODELS = [
    "openai/gpt-5.2",
    "anthropic/claude-opus-4-6",
]

# --- Test parameters ---
N_RUNS = 3           # Runs per model
TIMEOUT = 150        # seconds — generous to distinguish slow from broken
TEMPERATURE = 0.3

# --- Test question (hardcoded for reproducibility) ---
QUESTION_TEXT = "What will the percent change in US employment share of janitors and cleaners be from 2025 to 2030?"
RESOLUTION_CRITERIA = ""
FINE_PRINT = ""

# --- External research file (same as 013a) ---
NEWS_FILE = "../data/Run Log News Summaries/42562_News_Summary_03-15-2026.txt"

print(f"Models: {MODELS}")
print(f"Runs per model: {N_RUNS}")
print(f"Timeout: {TIMEOUT}s")

Models: ['openai/gpt-5.2', 'anthropic/claude-opus-4-6']
Runs per model: 3
Timeout: 150s


In [2]:
# Setup

import os
import asyncio
import time

import nest_asyncio
from openai import AsyncOpenAI

nest_asyncio.apply()

client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# Load research context
with open(NEWS_FILE, encoding='utf-8') as f:
    research = f.read()

# Build prompt (same as 013a::_build_base_rate_prompt)
PROMPT = (
    "You are a base rate analyst for a professional forecasting team.\n"
    "Given the following question and any available research context, "
    "provide a detailed base rate analysis:\n\n"
    "1. Identify the most relevant reference class(es)\n"
    "2. Estimate historical frequency: how often has this type of event occurred?\n"
    "3. Provide the numerator (events of interest) and denominator (opportunities)\n"
    "4. Note any trend (increasing, decreasing, stable)\n"
    "5. State key caveats or differences between the reference class and "
    "this specific question\n\n"
    f"Question: {QUESTION_TEXT}\n"
    f"\nRelevant news and research:\n{research}"
)

print(f"Prompt length: {len(PROMPT)} chars")
print(f"Research length: {len(research)} chars")
print("Ready.")

Prompt length: 15288 chars
Research length: 14632 chars
Ready.


In [3]:
# Run latency test

results = []  # list of {model, run, elapsed_s, chars, status}

for model_base in MODELS:
    model = model_base + ":online"
    print(f"\n{'='*60}")
    print(f"{model} — {N_RUNS} runs")
    print(f"{'='*60}")
    
    for run in range(1, N_RUNS + 1):
        print(f"  Run {run}/{N_RUNS}...", end=" ", flush=True)
        start = time.time()
        try:
            response = await asyncio.wait_for(
                client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": PROMPT}],
                    temperature=TEMPERATURE,
                ),
                timeout=TIMEOUT,
            )
            elapsed = time.time() - start
            content = response.choices[0].message.content
            chars = len(content)
            status = "OK"
            print(f"{status} — {elapsed:.1f}s, {chars} chars")
        except TimeoutError:
            elapsed = time.time() - start
            chars = 0
            status = "TIMEOUT"
            print(f"{status} — {elapsed:.1f}s")
        except Exception as e:
            elapsed = time.time() - start
            chars = 0
            status = f"ERROR: {e}"
            print(f"{status} — {elapsed:.1f}s")
        
        results.append({
            'model': model,
            'run': run,
            'elapsed_s': elapsed,
            'chars': chars,
            'status': status,
        })

print(f"\nDone. {len(results)} total runs.")


openai/gpt-5.2:online — 3 runs
TIMEOUT — 150.4s
TIMEOUT — 150.0s
TIMEOUT — 150.0s

anthropic/claude-opus-4-6:online — 3 runs
OK — 63.2s, 7206 chars
OK — 72.3s, 8322 chars
OK — 59.3s, 6600 chars

Done. 6 total runs.


In [4]:
# Summary table

print(f"{'Model':<40} {'Run':>4} {'Time (s)':>9} {'Chars':>7} {'Status':<10}")
print("-" * 75)
for r in results:
    print(f"{r['model']:<40} {r['run']:>4} {r['elapsed_s']:>9.1f} {r['chars']:>7} {r['status']:<10}")

# Per-model stats
print(f"\n{'='*75}")
print("Per-model summary (successful runs only):")
print(f"{'='*75}")

for model_base in MODELS:
    model = model_base + ":online"
    runs = [r for r in results if r['model'] == model]
    ok_runs = [r for r in runs if r['status'] == 'OK']
    timeouts = [r for r in runs if r['status'] == 'TIMEOUT']
    
    print(f"\n{model}:")
    print(f"  Success: {len(ok_runs)}/{len(runs)}")
    print(f"  Timeouts: {len(timeouts)}/{len(runs)}")
    if ok_runs:
        times = [r['elapsed_s'] for r in ok_runs]
        print(f"  Latency: min={min(times):.1f}s, max={max(times):.1f}s, mean={sum(times)/len(times):.1f}s")
        chars = [r['chars'] for r in ok_runs]
        print(f"  Output:  min={min(chars)}, max={max(chars)}, mean={sum(chars)//len(chars)}")
    else:
        print(f"  No successful runs")

print(f"\nTimeout threshold: {TIMEOUT}s")

Model                                     Run  Time (s)   Chars Status    
---------------------------------------------------------------------------
openai/gpt-5.2:online                       1     150.4       0 TIMEOUT   
openai/gpt-5.2:online                       2     150.0       0 TIMEOUT   
openai/gpt-5.2:online                       3     150.0       0 TIMEOUT   
anthropic/claude-opus-4-6:online            1      63.2    7206 OK        
anthropic/claude-opus-4-6:online            2      72.3    8322 OK        
anthropic/claude-opus-4-6:online            3      59.3    6600 OK        

Per-model summary (successful runs only):

openai/gpt-5.2:online:
  Success: 0/3
  Timeouts: 3/3
  No successful runs

anthropic/claude-opus-4-6:online:
  Success: 3/3
  Timeouts: 0/3
  Latency: min=59.3s, max=72.3s, mean=64.9s
  Output:  min=6600, max=8322, mean=7376

Timeout threshold: 150s
